# 02. Provider features

Goal: build one table, `provider_features`, with one row per NPI and only the columns a fraud
model should see. Inputs, not verdicts.

Three design rules:

1. **Ratios, not totals.** Raw totals mostly measure practice size. A large honest group beats
   a small fraudulent one on every raw column. Size-independent ratios are what carry signal.
2. **Compare within peers.** Forty services per patient is normal for a dermatologist and
   wild for an internist. Every feature gets a z-score computed inside its provider type.
3. **Leave out what you cannot defend.** Names, addresses, race and sex counts, and the 26
   chronic-condition percentages stay out. The conditions are already summarized by the risk
   score, and 26 noisy columns against 68 positives is how you overfit.

Grouping keys (provider type, state, entity code) are kept for comparison and later
drill-down, but they are not model inputs in their raw form.

In [10]:
import duckdb
import pandas as pd

DB_PATH = "C:/Users/palla/OneDrive/Documents/Coding Projects/Medicare Fraud ML/database/medicare_fraud.duckdb"
con = duckdb.connect(DB_PATH)

def q(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()

## 1. Features from the provider table

Every provider in the file has at least 11 beneficiaries (CMS suppresses anything smaller),
so the per-beneficiary divisions are safe.

| feature | formula | why it matters |
|---|---|---|
| `srvcs_per_bene` | services / beneficiaries | Intensity per patient. Already the clearest signal in the raw medians (5.1 vs 2.9) |
| `pymt_per_bene` | Medicare paid / beneficiaries | Dollar intensity per patient |
| `chrg_to_alowd` | submitted charge / allowed amount | Providers billing 10x the fee schedule are a known pattern |
| `drug_pymt_share` | drug payment / total payment | Infusion and injectable fraud is its own category |
| `risk_score` | average HCC risk score | 1.0 is national average. Healthy panel plus heavy billing is the classic mismatch |
| `pymt_per_bene_per_risk` | pymt_per_bene / risk_score | Billing not explained by how sick the patients are |
| `pct_under65` | under-65 beneficiaries / beneficiaries | Disability and ESRD populations, frequently targeted |
| `pct_dual` | dual-eligible / beneficiaries | Low-income populations, frequently targeted |
| `avg_age` | mean beneficiary age | Panel plausibility |
| `n_distinct_codes` | distinct HCPCS codes billed | Breadth. Very narrow or very broad for the specialty is unusual |
| `tot_benes` | beneficiaries | Size. Kept so the model can learn that some ratios behave differently at scale |

Drug totals are suppressed (null) for ~150K providers who gave drugs to fewer than 11 patients.
That means "small", so `drug_pymt_share` treats null as zero. The same goes for the
age and dual-eligibility counts. Eight providers were paid nothing at all in 2024, which would
make the share 0/0; those get zero as well.

In [11]:
con.execute("""
CREATE OR REPLACE TABLE feat_provider AS
SELECT
    Rndrng_NPI                                         AS npi,
    Rndrng_Prvdr_Type                                  AS provider_type,
    Rndrng_Prvdr_State_Abrvtn                          AS state,
    Rndrng_Prvdr_Ent_Cd                                AS entity_code,
    Tot_Benes                                          AS tot_benes,
    Tot_HCPCS_Cds                                      AS n_distinct_codes,
    Tot_Srvcs / Tot_Benes                              AS srvcs_per_bene,
    Tot_Mdcr_Pymt_Amt / Tot_Benes                      AS pymt_per_bene,
    Tot_Sbmtd_Chrg / Tot_Mdcr_Alowd_Amt                AS chrg_to_alowd,
    coalesce(coalesce(Drug_Mdcr_Pymt_Amt, 0) / nullif(Tot_Mdcr_Pymt_Amt, 0), 0) AS drug_pymt_share,
    Bene_Avg_Risk_Scre                                 AS risk_score,
    (Tot_Mdcr_Pymt_Amt / Tot_Benes) / Bene_Avg_Risk_Scre AS pymt_per_bene_per_risk,
    coalesce(Bene_Age_LT_65_Cnt, 0) / Tot_Benes        AS pct_under65,
    coalesce(Bene_Dual_Cnt, 0) / Tot_Benes             AS pct_dual,
    Bene_Avg_Age                                       AS avg_age
FROM provider
""")
q("SELECT count(*) AS n FROM feat_provider")

,n
0,1296739


## 2. Features from the provider_service table

This is the 9.8M row table. We aggregate it down to one row per NPI in SQL and never load it
into pandas.

| feature | formula | why it matters |
|---|---|---|
| `sameday_repeat_ratio` | services / beneficiary-day services, non-drug rows only | Above 1.0 means the same code billed more than once per patient per day |
| `em_high_share` | 99214 + 99215 services / 99212 through 99215 services | Upcoding office visits. Null for providers who bill no office visits |
| `top_code_pymt_share` | payment from the single most-paid code / total payment | Fraud rings often run one code at volume |
| `facility_share` | facility services / all services | Place-of-service manipulation |
| `n_service_rows` | count of NPI x code x place rows | Breadth, finer than distinct codes |

Two traps here. Drug rows count dose units as "services", so a single infusion can be 400
services on one patient-day. The same-day ratio is computed on non-drug rows only. And 99211
was effectively retired in 2021, so the E&M denominator starts at 99212.

Only 1.21M of 1.30M providers appear in this table at all. The other ~90K had every
code-level row suppressed. Their service features will be null and the model must tolerate that.

In [12]:
con.execute("""
CREATE OR REPLACE TABLE feat_service AS
SELECT
    Rndrng_NPI AS npi,
    count(*)   AS n_service_rows,

    sum(CASE WHEN HCPCS_Drug_Ind = 'N' THEN Tot_Srvcs END)
      / nullif(sum(CASE WHEN HCPCS_Drug_Ind = 'N' THEN Tot_Bene_Day_Srvcs END), 0)
      AS sameday_repeat_ratio,

    sum(CASE WHEN HCPCS_Cd IN ('99214','99215') THEN Tot_Srvcs END)
      / nullif(sum(CASE WHEN HCPCS_Cd IN ('99212','99213','99214','99215') THEN Tot_Srvcs END), 0)
      AS em_high_share,

    max(Tot_Srvcs * Avg_Mdcr_Pymt_Amt) / nullif(sum(Tot_Srvcs * Avg_Mdcr_Pymt_Amt), 0)
      AS top_code_pymt_share,

    coalesce(sum(CASE WHEN Place_Of_Srvc = 'F' THEN Tot_Srvcs END), 0) / sum(Tot_Srvcs)
      AS facility_share
FROM provider_service
GROUP BY Rndrng_NPI
""")
q("SELECT count(*) AS n, sum(em_high_share IS NULL) AS em_null FROM feat_service")

,n,em_null
0,1207473,695975.0


## 3. Peer z-scores

A z-score says how many spreads a value sits from the center of its group. The ordinary
version uses mean and standard deviation, but billing data has heavy right tails: a handful of
providers billing millions would drag the mean and inflate the standard deviation for everyone.

So we use the **robust** version. Center is the median. Spread is the median absolute deviation
(MAD), scaled by 1.4826 so that it equals the standard deviation when the data happen to be normal.

```
z = (x - median_group) / (1.4826 * MAD_group)
```

Groups are provider types. 20 of the 113 types have fewer than 100 members, too few for a
stable median and MAD, so those fall back to the global statistics.

MAD has one failure mode. If more than half the providers share the same value (drug share is
zero for most, facility share is exactly 1.0 for most), the MAD is zero and the z-score divides
by nothing. For those features the spread falls back to the ordinary standard deviation, which
is never zero unless the whole column is constant. The fallback chain is: group MAD, then global
MAD, then group standard deviation, then global standard deviation.

The SQL below is generated from a list of feature names rather than written 15 times by hand.
Print `sql` if you want to read the expanded version.

In [13]:
FEATURES = [
    "srvcs_per_bene", "pymt_per_bene", "chrg_to_alowd", "drug_pymt_share",
    "risk_score", "pymt_per_bene_per_risk", "pct_under65", "pct_dual", "avg_age",
    "n_distinct_codes", "tot_benes",
    "sameday_repeat_ratio", "em_high_share", "top_code_pymt_share", "facility_share",
    "n_service_rows",
]
MIN_GROUP = 100
K = 1.4826

stat_cols = ",\n    ".join(
    f"median({f}) AS med_{f}, mad({f}) AS mad_{f}, stddev_samp({f}::DOUBLE) AS sd_{f}"
    for f in FEATURES
)
z_cols = ",\n    ".join(
    f"(b.{f} - coalesce(g.med_{f}, a.med_{f})) / coalesce("
    f"nullif({K} * g.mad_{f}, 0), nullif({K} * a.mad_{f}, 0), "
    f"nullif(g.sd_{f}, 0), nullif(a.sd_{f}, 0)) AS z_{f}"
    for f in FEATURES
)
raw_cols = ",\n    ".join(f"b.{f}" for f in FEATURES)

sql = f"""
CREATE OR REPLACE TABLE provider_features AS
WITH base AS (
    SELECT p.*, s.n_service_rows, s.sameday_repeat_ratio, s.em_high_share,
           s.top_code_pymt_share, s.facility_share,
           lb.label
    FROM feat_provider p
    LEFT JOIN feat_service s USING (npi)
    LEFT JOIN labels lb USING (npi)
),
grp AS (
    SELECT provider_type, count(*) AS n_in_group,
    {stat_cols}
    FROM base GROUP BY provider_type
    HAVING count(*) >= {MIN_GROUP}
),
alls AS (
    SELECT {stat_cols}
    FROM base
)
SELECT
    b.npi, b.provider_type, b.state, b.entity_code, b.label,
    coalesce(g.n_in_group, 0) >= {MIN_GROUP} AS has_peer_group,
    {raw_cols},
    {z_cols}
FROM base b
LEFT JOIN grp g USING (provider_type)
CROSS JOIN alls a
"""
con.execute(sql)
q("SELECT count(*) AS n, sum(has_peer_group) AS with_peers, count(*) - sum(has_peer_group) AS global_fallback FROM provider_features")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n,with_peers,global_fallback
0,1296739,1296259.0,480.0


## 4. Does the table look right?

Three checks. Nulls per column, so we know what the model has to tolerate. Then the raw medians
by label, as in the load notebook but now on the engineered features. Then the z-score medians
by label, which is the real test: after peer adjustment, do the positives still stand out?

A z-score median near 0 for the negatives is expected by construction. What we want to see is
the positives' median pushed away from 0 on at least a few features.

In [14]:
null_counts = q(f"""
SELECT {", ".join(f"sum({f} IS NULL) AS {f}" for f in FEATURES)}
FROM provider_features
""").T
null_counts.columns = ["n_null"]
null_counts

,n_null
srvcs_per_bene,0.0
pymt_per_bene,0.0
chrg_to_alowd,0.0
drug_pymt_share,0.0
risk_score,0.0
pymt_per_bene_per_risk,0.0
pct_under65,0.0
pct_dual,0.0
avg_age,0.0
n_distinct_codes,0.0


In [15]:
raw_by_label = q(f"""
SELECT label, count(*) AS n,
       {", ".join(f"round(median({f}), 3) AS {f}" for f in FEATURES)}
FROM provider_features
WHERE label IS NOT NULL
GROUP BY label ORDER BY label
""").set_index("label").T
raw_by_label

label,0,1
n,1296590.000,68.000
srvcs_per_bene,2.873,5.121
pymt_per_bene,187.848,269.459
chrg_to_alowd,2.851,1.876
drug_pymt_share,0.000,0.000
risk_score,1.355,1.301
pymt_per_bene_per_risk,126.870,191.034
pct_under65,0.000,0.000
pct_dual,0.080,0.107
avg_age,74.000,73.000


In [16]:
z_by_label = q(f"""
SELECT label,
       {", ".join(f"round(median(z_{f}), 2) AS z_{f}" for f in FEATURES)}
FROM provider_features
WHERE label IS NOT NULL
GROUP BY label ORDER BY label
""").set_index("label").T
z_by_label["gap"] = z_by_label[1] - z_by_label[0]
z_by_label.sort_values("gap", key=abs, ascending=False)

label,0,1,gap
z_srvcs_per_bene,0.0,0.56,0.56
z_chrg_to_alowd,0.0,-0.53,-0.53
z_em_high_share,0.0,0.52,0.52
z_pymt_per_bene_per_risk,0.0,0.27,0.27
z_n_distinct_codes,0.0,-0.25,-0.25
z_risk_score,0.0,-0.09,-0.09
z_tot_benes,0.0,-0.07,-0.07
z_pct_dual,0.0,0.04,0.04
z_top_code_pymt_share,0.0,-0.04,-0.04
z_pymt_per_bene,0.0,0.02,0.02


## 5. Clean up

The two intermediate tables are dropped. `provider_features` is the only thing later notebooks read.

In [17]:
con.execute("DROP TABLE feat_provider")
con.execute("DROP TABLE feat_service")
q("SELECT table_name, estimated_size AS approx_rows FROM duckdb_tables() ORDER BY 1")

,table_name,approx_rows
0,labels,1296739
1,leie,83842
2,provider,1296739
3,provider_features,1296739
4,provider_service,9781673


In [18]:
con.close()